# Method B 질적분석 — 최종 모델(exp10) 조합 검토

> **목적**: exp10(`learned_hop_sum` readout, test PR-AUC 0.684)이 뽑는 K-P-K 조합을
> 정성 검토해 ① 모델 품질 판정 ② **대시보드 네트워크 viz 설계 근거** 확보.
> 대시보드와 동일한 `serve.py` 로직 사용 → 화면 조합 = 분석 조합 일치.
>
> 선행: methodB 실험 → best=exp10 (`serve.SERVING_EXP`). 오프라인(parquet), Gemma 불필요.

## Phase 0. 셋업 & 시드 큐레이션

In [ ]:
import os, sys
from pathlib import Path
from collections import defaultdict
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, matplotlib
_anchor = Path(globals().get("__vsc_ipynb_file__", ".")).resolve().parent if globals().get("__vsc_ipynb_file__") else Path.cwd()
ROOT = next((str(p) for p in [_anchor, *_anchor.parents] if (p / "CLAUDE.md").exists()), str(_anchor))
sys.path.insert(0, ROOT); os.chdir(ROOT)
matplotlib.rcParams["font.family"] = "Malgun Gothic"; matplotlib.rcParams["axes.unicode_minus"] = False

from src.eval import serve
D = serve._data()
print("메인 모델(SERVING_EXP):", serve.SERVING_EXP)
print("제품", len(D["scores"]), "/ 키워드", len(D["graph_kw"]))

# 큐레이션 시드 (맛·재료·IP·트렌드 혼합) — 그래프에 존재하는 것만
CURATED = ["마라","로제","흑임자","단백질","크림","치즈","매콤","달콤","딸기","복숭아",
           "감자","띠부씰","콜라보","한정","제로","수제","두바이","흑백요리사"]
SEEDS = [s for s in CURATED if s in D["graph_kw"]][:15]
print("분석 시드:", SEEDS)

# 키워드별 제품 수(빈도) — rare-keyword 편향 진단용
kw_deg = defaultdict(int)
for kw, lst in [(k, v) for k, v in D["kw2prod"].items()]:
    kw_deg[kw] = len(lst)

def raw_cooc(seed, k=8):
    """원천 동시출현(가중치·성공 무시, 단순 count) — 학습 대비 베이스라인."""
    cnt = defaultdict(int)
    for prod, _ in D["kw2prod"].get(seed, []):
        for kt, _ in D["prod2kw"].get(prod, []):
            if kt != seed:
                cnt[kt] += 1
    return [k for k, _ in sorted(cnt.items(), key=lambda x: -x[1])[:k]]


## Phase 1. 시드별 조합 정성 검토 (학습 가중 K-P-K)

In [ ]:
# 시드별 학습 K-P-K 조합 top-6 + 경유 히트제품 (정성 판정용)
rows = []
for s in SEEDS:
    rec = serve.recommend_keywords([s], 6)
    via = serve._top_via([s])
    rows.append({"시드": s, "추천 조합(학습 가중)": ", ".join(k for k, _ in rec),
                 "경유 히트제품": via, "판정(수기)": ""})
df1 = pd.DataFrame(rows)
import IPython.display as ipd; ipd.display(df1)
print("\\n→ '판정(수기)' 열에 타당/애매/노이즈 직접 기입하며 검토")


## Phase 2. 학습 가중 vs 원천 동시출현 (학습이 가치를 더하나)

In [ ]:
# 학습 가중(K-P-K) vs 원천 동시출현(count) — 순위 차이·겹침
print(f"{'시드':<8}{'겹침(top6)':>10}   학습 only  /  원천 only")
print("-" * 70)
for s in SEEDS:
    learned = [k for k, _ in serve.recommend_keywords([s], 6)]
    raw = raw_cooc(s, 6)
    inter = set(learned) & set(raw)
    only_l = [k for k in learned if k not in raw]
    only_r = [k for k in raw if k not in learned]
    print(f"{s:<8}{len(inter)}/6{'':>6}   L:{','.join(only_l[:3]):<22} R:{','.join(only_r[:3])}")
print("\\n→ 겹침이 낮을수록 '학습이 단순 빈도와 다른 신호'를 잡은 것. only_L = 학습이 끌어올린 조합")


## Phase 3. 노이즈·편향 진단 (허브 지배 / rare-keyword 편향)

In [ ]:
# 노이즈·편향 진단
# (1) 허브 키워드 지배: 시드 전반 추천에 자주 등장하는 키워드
appear = defaultdict(int)
for s in SEEDS:
    for k, _ in serve.recommend_keywords([s], 8):
        appear[k] += 1
hub = sorted(appear.items(), key=lambda x: -x[1])[:12]
print("[허브 키워드] 여러 시드 추천에 반복 등장 (지배 = 노이즈 의심):")
for k, c in hub:
    print(f"  {k}: {c}/{len(SEEDS)} 시드  (제품 {kw_deg.get(k,0)}개 보유)")

# (2) rare-keyword 편향: 추천 상위 키워드의 제품 빈도 분포 (att은 키워드 기준 정규화 → 희귀 과대평가 경향)
recd = [k for s in SEEDS for k, _ in serve.recommend_keywords([s], 8)]
deg_recd = [kw_deg.get(k, 0) for k in recd]
print(f"\\n[추천 키워드 제품빈도] 중앙값 {int(np.median(deg_recd))} / 평균 {np.mean(deg_recd):.1f} "
      f"/ 전체 키워드 중앙값 {int(np.median(list(kw_deg.values())))}")
print("→ 추천 빈도가 전체보다 현저히 낮으면 rare-keyword 편향 (빈도 보정 필요)")
plt.figure(figsize=(7,3)); plt.hist(deg_recd, bins=30); plt.xlabel("추천 키워드의 제품 수"); plt.ylabel("빈도")
plt.title("추천된 키워드의 제품 빈도 분포 (왼쪽 치우침=희귀 편향)"); plt.tight_layout(); plt.show()


## Phase 4. 조합 서브그래프 추출 → 대시보드 viz 설계 입력

In [ ]:
# 조합 서브그래프 추출 + 시각화 (대시보드 viz 설계 입력)
SEED_EX = SEEDS[0]
props = serve.recommend_proposals("삼각김밥", serve.infer_attrs(SEED_EX) or [SEED_EX], SEED_EX, k=1, use_rag=False)
p = props[0]
print("예시 조합:", p["name"], "| attrs:", p["attrs"], "| 경유:", p["via"])

# 엣지 추출: 경유 제품의 (키워드, attention) — 실제 대시보드가 그릴 엣지
via_name = p["via"].replace("기존 ", "")
edges = sorted(D["prod2kw"].get(via_name, []), key=lambda x: -x[1])[:10]
edf = pd.DataFrame(edges, columns=["키워드", "attention(학습가중)"])
import IPython.display as ipd; ipd.display(edf)

try:
    import networkx as nx
    G = nx.Graph()
    G.add_node(p["name"], kind="product")
    for kw, w in edges:
        G.add_node(kw, kind="kw"); G.add_edge(p["name"], kw, weight=w)
    pos = nx.spring_layout(G, seed=42)
    plt.figure(figsize=(8,5))
    nx.draw_networkx_nodes(G, pos, node_color=["#00C896" if G.nodes[n]["kind"]=="product" else "#4A9EFF" for n in G],
                           node_size=[1400 if G.nodes[n]["kind"]=="product" else 700 for n in G])
    ws = [G[u][v]["weight"] for u, v in G.edges()]
    nx.draw_networkx_edges(G, pos, width=[1+8*w/max(ws) for w in ws], alpha=0.5)
    nx.draw_networkx_labels(G, pos, font_family="Malgun Gothic", font_size=9)
    plt.title(f"조합 서브그래프: {p['name']} (엣지 두께=학습 attention)"); plt.axis("off"); plt.tight_layout(); plt.show()
except ImportError:
    print("networkx 없음 → 위 엣지 표로 대체")
print("\\n→ viz 설계: 노드 수 / attention 임계(약한 엣지 숨김) / 경유제품·IP 포함 여부를 여기서 결정")


## Phase 5. IP·카테고리 단면

In [ ]:
# IP 단면 + 카테고리 단면
# IP: 제품-IP 엣지 로드 → 상위 IP의 연결 제품/키워드
pi = pd.read_parquet("data/processed/hin/product_ip_edges.parquet")
top_ip = pi["ip_name"].value_counts().head(8)
print("[제품 많이 보유한 IP top8]:")
print(top_ip.to_string())

# 카테고리 단면: 같은 시드, 카테고리만 바꿔 → 조합 동일한지(현재 라벨-only 설계 확인)
s = SEEDS[0]; attrs = serve.infer_attrs(s) or [s]
print(f"\\n[카테고리 단면 — 시드 '{s}', 카테고리만 변경]")
for cat in ["삼각김밥", "도시락", "냉장간편식"]:
    names = [pp["name"] for pp in serve.recommend_proposals(cat, attrs, s, k=2, use_rag=False)]
    print(f"  {cat}: {names}")
print("→ 키워드 조합이 동일하면 현재 카테고리는 라벨-only (순회 미제약). 신규 기능 시 카테고리 필터 검토")


## Phase 6. 정성 결론 (수기 작성)

### 1. 조합 품질 (Phase 1·2)
- 타당한 시드: _____
- 애매/노이즈 시드: _____
- 학습 vs 원천: 학습이 가치를 더하는가? (only_L 조합이 더 타당한가) _____

### 2. 노이즈·편향 (Phase 3)
- 허브 키워드 지배 여부: _____
- rare-keyword 편향 여부 / 빈도 보정 필요성: _____

### 3. 대시보드 네트워크 viz 사양 (Phase 4) ★
- 한 조합당 노드 수: _____
- attention 임계(약한 엣지 컷): _____
- 경유 히트제품 노드 포함? / IP 노드 포함? _____
- 강조 기준(색·크기): 학습 attention / 성공확률 _____

### 4. 모델·설계 개선점
- 카테고리 제약 도입 필요? (Phase 5) _____
- 기타: _____
